### 1. Libraries

In [1]:
# Data handling
import numpy as np
import pandas as pd

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

### 2. Data Loading

In [2]:
df_train = pd.read_csv("data/train.csv")
df_test = pd.read_csv("data/test.csv")

### 3. Feature Engineering and Model Training

In [3]:
target_col = 'Churn'

#### 3.1. Logistic Regression

In [4]:
df_train_log = df_train.copy()
df_train_log[target_col] = df_train_log[target_col].map({"No": 0, "Yes": 1})

In [5]:
df_test_log = df_test.copy()

##### 3.1.1. Feature Engineering

**Household Type**

When customer have either `Partner` or `Dependent` or both, the churn rates significantly go down, possibly because of more sharing of the service, leading to higher number of users per household, and thus loyalty. 

In [6]:
def get_household_type(row):
    if row['Partner'] == 'No' and row['Dependents'] == 'No':
        return 'Single'
    elif row['Partner'] == 'Yes' and row['Dependents'] == 'No':
        return 'Couple'
    elif row['Partner'] == 'Yes' and row['Dependents'] == 'Yes':
        return 'Family'
    else:
        return 'Single Parent'

df_train_log['household_type'] = df_train_log.apply(get_household_type, axis=1)
df_test_log['household_type'] = df_test_log.apply(get_household_type, axis=1)

**Service Engagement Score**

The more protection services a customer have, the less the churn rate. 

In [7]:
services = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]

df_train_log["num_services"] = df_train_log[services].apply(lambda x: (x == "Yes").sum(), axis=1)
df_test_log["num_services"] = df_test_log[services].apply(lambda x: (x == "Yes").sum(), axis=1)

**High Churn Rate Signal Group**

In [8]:
# Create the specific high-risk flag
df_train_log['high_risk_digital_senior'] = (
    (df_train_log['SeniorCitizen'] == 1) & 
    (df_train_log['PaymentMethod'] == 'Electronic check') & 
    (df_train_log['PaperlessBilling'] == 'Yes')
).astype(int)

# Apply the same flag to the test set
df_test_log['high_risk_digital_senior'] = (
    (df_test_log['SeniorCitizen'] == 1) & 
    (df_test_log['PaymentMethod'] == 'Electronic check') & 
    (df_test_log['PaperlessBilling'] == 'Yes')
).astype(int)

##### 3.1.2. Model Training

In [9]:
X = df_train_log.drop(columns=[target_col, "id"]).copy()
y = df_train_log[target_col].copy()
X_test = df_test_log.drop(columns=["id"]).copy()

In [10]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

In [11]:
preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ]
)

preprocessor_unscaled = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ]
)

In [12]:
def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

In [13]:
from copy import deepcopy

def run_oof(model, X, y, X_test, cv, model_name="model"):
    oof_pred = np.zeros(len(X))
    test_pred_folds = np.zeros((len(X_test), cv.get_n_splits()))
    fold_metrics = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_train_fold = X.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]
        y_valid_fold = y.iloc[valid_idx]

        model_fold = deepcopy(model)
        model_fold.fit(X_train_fold, y_train_fold)

        valid_prob = model_fold.predict_proba(X_valid_fold)[:, 1]
        test_prob = model_fold.predict_proba(X_test)[:, 1]

        oof_pred[valid_idx] = valid_prob
        test_pred_folds[:, fold - 1] = test_prob

        fold_result = evaluate_predictions(y_valid_fold, valid_prob, threshold=0.5)
        fold_result["fold"] = fold
        fold_metrics.append(fold_result)

        print(
            f"{model_name} | Fold {fold} | "
            f"AUC={fold_result['roc_auc']:.4f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"Precision={fold_result['precision']:.4f} | "
            f"Recall={fold_result['recall']:.4f}"
        )

    fold_metrics_df = pd.DataFrame(fold_metrics)
    overall_metrics = evaluate_predictions(y, oof_pred, threshold=0.5)

    print(f"\n{model_name} OOF Overall:")
    for k, v in overall_metrics.items():
        print(f"{k}: {v:.4f}")

    return {
        "oof_pred": oof_pred,
        "test_pred": test_pred_folds.mean(axis=1),
        "fold_metrics": fold_metrics_df,
        "overall_metrics": overall_metrics
    }

In [14]:
def find_best_threshold(y_true, y_prob, metric="f1", thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.01)

    rows = []
    for t in thresholds:
        scores = evaluate_predictions(y_true, y_prob, threshold=t)
        scores["threshold"] = t
        rows.append(scores)

    result_df = pd.DataFrame(rows)
    best_row = result_df.sort_values(metric, ascending=False).iloc[0]
    return result_df, best_row

In [15]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [16]:
from sklearn.linear_model import LogisticRegression

In [17]:
logreg_candidates = {
    "logreg_l2_stronger_reg": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(
            penalty="l2",
            C=0.3,
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        ))
    ])
}

In [18]:
logreg_results = {}

for name, model in logreg_candidates.items():
    print(f"\n{'='*70}\nRunning {name}\n{'='*70}")
    result = run_oof(model, X, y, X_test, cv, model_name=name)
    logreg_results[name] = result


Running logreg_l2_stronger_reg
logreg_l2_stronger_reg | Fold 1 | AUC=0.9074 | F1=0.6737 | Precision=0.5458 | Recall=0.8801
logreg_l2_stronger_reg | Fold 2 | AUC=0.9088 | F1=0.6715 | Precision=0.5418 | Recall=0.8829
logreg_l2_stronger_reg | Fold 3 | AUC=0.9080 | F1=0.6720 | Precision=0.5438 | Recall=0.8792
logreg_l2_stronger_reg | Fold 4 | AUC=0.9090 | F1=0.6748 | Precision=0.5476 | Recall=0.8791
logreg_l2_stronger_reg | Fold 5 | AUC=0.9059 | F1=0.6692 | Precision=0.5416 | Recall=0.8754

logreg_l2_stronger_reg OOF Overall:
accuracy: 0.8069
precision: 0.5441
recall: 0.8793
f1: 0.6722
roc_auc: 0.9078


In [19]:
logreg_summary = pd.DataFrame({
    name: result["overall_metrics"]
    for name, result in logreg_results.items()
}).T.sort_values("roc_auc", ascending=False)

logreg_summary

,accuracy,precision,recall,f1,roc_auc
logreg_l2_stronger_reg,0.806895,0.544102,0.879313,0.672237,0.907791


In [20]:
best_logreg_name = logreg_summary.index[0]
best_logreg_oof = logreg_results[best_logreg_name]["oof_pred"]
best_logreg_test = logreg_results[best_logreg_name]["test_pred"]

In [21]:
threshold_df, best_threshold_row = find_best_threshold(y, best_logreg_oof, metric="f1")
best_threshold_row

accuracy     0.840221
precision    0.613024
recall       0.787874
f1           0.689538
roc_auc      0.907791
threshold    0.660000
Name: 61, dtype: float64

#### 3.2. XGBoost

In [22]:
from xgboost import XGBClassifier

In [23]:
'''
xgb_base = Pipeline([
    ("preprocessor", preprocessor_unscaled),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        min_child_weight=1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])
'''

'\nxgb_base = Pipeline([\n    ("preprocessor", preprocessor_unscaled),\n    ("model", XGBClassifier(\n        n_estimators=300,\n        max_depth=4,\n        learning_rate=0.05,\n        subsample=0.8,\n        colsample_bytree=0.8,\n        reg_alpha=0.0,\n        reg_lambda=1.0,\n        min_child_weight=1,\n        objective="binary:logistic",\n        eval_metric="logloss",\n        random_state=42,\n        n_jobs=-1\n    ))\n])\n'

In [24]:
#xgb_result = run_oof(xgb_base, X, y, X_test, cv, model_name="xgb_base")

In [25]:
xgb_candidates = {
    "xgb_stronger_reg": Pipeline([
        ("preprocessor", preprocessor_unscaled),
        ("model", XGBClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.8,
            reg_alpha=0.2,
            reg_lambda=2.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

In [26]:
xgb_results = {}

for name, model in xgb_candidates.items():
    print(f"\n{'='*70}\nRunning {name}\n{'='*70}")
    result = run_oof(model, X, y, X_test, cv, model_name=name)
    xgb_results[name] = result


Running xgb_stronger_reg
xgb_stronger_reg | Fold 1 | AUC=0.9147 | F1=0.6712 | Precision=0.7092 | Recall=0.6371
xgb_stronger_reg | Fold 2 | AUC=0.9158 | F1=0.6748 | Precision=0.7086 | Recall=0.6440
xgb_stronger_reg | Fold 3 | AUC=0.9151 | F1=0.6738 | Precision=0.7103 | Recall=0.6408
xgb_stronger_reg | Fold 4 | AUC=0.9163 | F1=0.6737 | Precision=0.7147 | Recall=0.6370
xgb_stronger_reg | Fold 5 | AUC=0.9134 | F1=0.6722 | Precision=0.7096 | Recall=0.6385

xgb_stronger_reg OOF Overall:
accuracy: 0.8601
precision: 0.7105
recall: 0.6395
f1: 0.6731
roc_auc: 0.9150


In [27]:
xgb_summary = pd.DataFrame({
    name: result["overall_metrics"]
    for name, result in xgb_results.items()
}).T.sort_values("roc_auc", ascending=False)

xgb_summary

,accuracy,precision,recall,f1,roc_auc
xgb_stronger_reg,0.860123,0.710478,0.639493,0.673119,0.915039


In [28]:
best_xgb_name = xgb_summary.index[0]
best_xgb_oof = xgb_results[best_xgb_name]["oof_pred"]
best_xgb_test = xgb_results[best_xgb_name]["test_pred"]

#### 3.3. LightGBM

In [29]:
from lightgbm import LGBMClassifier

In [30]:
lgbm_candidates = {
    "lgbm_base": Pipeline([
        ("preprocessor", preprocessor_unscaled),
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ))
    ]),
    
    "lgbm_regularized": Pipeline([
        ("preprocessor", preprocessor_unscaled),
        ("model", LGBMClassifier(
            n_estimators=400,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            min_child_samples=20,
            random_state=42
        ))
    ]),
    
    "lgbm_deeper": Pipeline([
        ("preprocessor", preprocessor_unscaled),
        ("model", LGBMClassifier(
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=8,
            subsample=0.85,
            colsample_bytree=0.8,
            random_state=42
        ))
    ])
}

In [31]:
lgbm_results = {}

for name, model in lgbm_candidates.items():
    print(f"\n{'='*70}\nRunning {name}\n{'='*70}")
    result = run_oof(model, X, y, X_test, cv, model_name=name)
    lgbm_results[name] = result


Running lgbm_base
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008738 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 682
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567


/home/manyyou/miniconda3/envs/churn-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/manyyou/miniconda3/envs/churn-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm_base | Fold 1 | AUC=0.9154 | F1=0.6734 | Precision=0.7116 | Recall=0.6391
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008560 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 682
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567


/home/manyyou/miniconda3/envs/churn-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/manyyou/miniconda3/envs/churn-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm_base | Fold 2 | AUC=0.9164 | F1=0.6757 | Precision=0.7111 | Recall=0.6437
[LightGBM] [Info] Number of positive: 107053, number of negative: 368302
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023389 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 682
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225206 -> initscore=-1.235579
[LightGBM] [Info] Start training from score -1.235579


...

lgbm_deeper | Fold 3 | AUC=0.9160 | F1=0.6761 | Precision=0.7119 | Recall=0.6437
[LightGBM] [Info] Number of positive: 107053, number of negative: 368302
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020513 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 682
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225206 -> initscore=-1.235579
[LightGBM] [Info] Start training from score -1.235579


/home/manyyou/miniconda3/envs/churn-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/manyyou/miniconda3/envs/churn-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm_deeper | Fold 4 | AUC=0.9171 | F1=0.6770 | Precision=0.7171 | Recall=0.6410
[LightGBM] [Info] Number of positive: 107054, number of negative: 368302
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009060 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 682
[LightGBM] [Info] Number of data points in the train set: 475356, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235570
[LightGBM] [Info] Start training from score -1.235570
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/home/manyyou/miniconda3/envs/churn-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/manyyou/miniconda3/envs/churn-ml/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm_deeper | Fold 5 | AUC=0.9145 | F1=0.6771 | Precision=0.7124 | Recall=0.6451

lgbm_deeper OOF Overall:
accuracy: 0.8612
precision: 0.7126
recall: 0.6430
f1: 0.6760
roc_auc: 0.9160


In [32]:
lgbm_summary = pd.DataFrame({
    name: result["overall_metrics"]
    for name, result in lgbm_results.items()
}).T.sort_values("roc_auc", ascending=False)

lgbm_summary

,accuracy,precision,recall,f1,roc_auc
lgbm_deeper,0.861188,0.712557,0.643020,0.676005,0.915971
lgbm_base,0.860978,0.712764,0.641017,0.674989,0.915705
lgbm_regularized,0.860594,0.711855,0.640083,0.674064,0.915458


In [33]:
best_lgbm_name = lgbm_summary.index[0]
best_lgbm_oof = lgbm_results[best_lgbm_name]["oof_pred"]
best_lgbm_test = lgbm_results[best_lgbm_name]["test_pred"]

#### 3.4. Compare

In [34]:
base_model_summary = pd.DataFrame({
    "logreg": logreg_results[best_logreg_name]["overall_metrics"],
    "xgb": xgb_results[best_xgb_name]["overall_metrics"],
    "lgbm": lgbm_results[best_lgbm_name]["overall_metrics"]
}).T.sort_values("roc_auc", ascending=False)

base_model_summary

,accuracy,precision,recall,f1,roc_auc
lgbm,0.861188,0.712557,0.643020,0.676005,0.915971
xgb,0.860123,0.710478,0.639493,0.673119,0.915039
logreg,0.806895,0.544102,0.879313,0.672237,0.907791


In [35]:
oof_compare = pd.DataFrame({
    "logreg": best_logreg_oof,
    "xgb": best_xgb_oof,
    "lgbm": best_lgbm_oof
})

oof_compare.corr()

,logreg,xgb,lgbm
logreg,1.000000,0.938013,0.932292
xgb,0.938013,1.000000,0.996651
lgbm,0.932292,0.996651,1.000000


In [36]:
ensemble_oof_weighted = (
    0.2 * best_logreg_oof +
    0.4 * best_xgb_oof +
    0.4 * best_lgbm_oof
)

ensemble_test_weighted = (
    0.2 * best_logreg_test +
    0.4 * best_xgb_test +
    0.4 * best_lgbm_test
)

evaluate_predictions(y, ensemble_oof_weighted, threshold=0.5)

{'accuracy': 0.8584132455056765,
 'precision': 0.6773332381598202,
 'recall': 0.7091102027395623,
 'f1': 0.6928575600922924,
 'roc_auc': 0.9152094472197808}

In [37]:
meta_X_train = pd.DataFrame({
    "logreg_pred": best_logreg_oof,
    "xgb_pred": best_xgb_oof,
    "lgbm_pred": best_lgbm_oof
})

meta_X_test = pd.DataFrame({
    "logreg_pred": best_logreg_test,
    "xgb_pred": best_xgb_test,
    "lgbm_pred": best_lgbm_test
})

In [38]:
meta_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="liblinear",
    max_iter=2000,
    random_state=42
)

meta_model.fit(meta_X_train, y)

stack_oof = meta_model.predict_proba(meta_X_train)[:, 1]
stack_test = meta_model.predict_proba(meta_X_test)[:, 1]

evaluate_predictions(y, stack_oof, threshold=0.5)

{'accuracy': 0.8603032006381754,
 'precision': 0.7010525482747705,
 'recall': 0.6619861452580763,
 'f1': 0.680959500647636,
 'roc_auc': 0.9145597312307447}

In [39]:
test_ids = df_test["id"].copy()

In [40]:
submission = pd.DataFrame({
    "id": test_ids,
    "Churn": stack_test
})

submission.to_csv("submission.csv", index=False)
submission.head()

,id,Churn
0,594194,0.039793
1,594195,0.020935
2,594196,0.071462
3,594197,0.021853
4,594198,0.508006
